In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OI_RAW_DIR = Path("../user_data/data/gmx/open_interest/arbitrum/raw")
SCALE_30 = 10 ** 30  # GMX 30-decimal fixed-point

## Load & Aggregate OI Data

Reads all `OpenInterestUpdated` Parquet files, computes daily end-of-day long/short OI per market.
Alt-collateral bracket suffixes (e.g. `ETH/USD [USDC]`) are stripped so all ETH/USD variants merge.

In [ ]:
frames = []
for symbol_dir in sorted(OI_RAW_DIR.iterdir()):
    if not symbol_dir.is_dir():
        continue
    for f in symbol_dir.rglob("*.parquet"):
        frames.append(pd.read_parquet(f))

raw = pd.concat(frames, ignore_index=True)
oi = raw[raw["eventType"] == "OpenInterestUpdated"].copy()
oi["ts"] = pd.to_datetime(oi["blockTimestamp"], unit="s", utc=True)
oi["date"] = oi["ts"].dt.normalize()
oi["next_value_usd"] = oi["nextValueUsd"].astype(float) / SCALE_30
oi["symbol"] = oi["symbol"].str.replace(r"\s*\[.*\]", "", regex=True).str.strip()

daily = (
    oi.sort_values("ts")
    .groupby(["date", "symbol", "isLong"])["next_value_usd"]
    .last()
    .reset_index()
)
daily_long  = daily[daily["isLong"]].rename(columns={"next_value_usd": "oi_long"}).drop(columns="isLong")
daily_short = daily[~daily["isLong"]].rename(columns={"next_value_usd": "oi_short"}).drop(columns="isLong")
daily_oi = daily_long.merge(daily_short, on=["date", "symbol"], how="outer").fillna(0)
daily_oi["oi_total"] = daily_oi["oi_long"] + daily_oi["oi_short"]
daily_oi["oi_imbalance"] = daily_oi["oi_long"] - daily_oi["oi_short"]

print(f"Markets: {sorted(daily_oi['symbol'].unique())}")
print(f"Date range: {daily_oi['date'].min()} \u2192 {daily_oi['date'].max()}")
daily_oi.head()

## Chart 1: Total OI Over Time — Top 10 Markets

In [ ]:
top10 = (
    daily_oi.groupby("symbol")["oi_total"]
    .mean()
    .nlargest(10)
    .index.tolist()
)
df_top = daily_oi[daily_oi["symbol"].isin(top10)]

fig = px.line(
    df_top,
    x="date", y="oi_total", color="symbol",
    title="GMX V2 — Total Open Interest (USD) — Top 10 Markets",
    labels={"oi_total": "OI (USD)", "date": "Date"},
    template="plotly_dark",
)
fig.update_layout(hovermode="x unified")
fig.show()

## Chart 2: Long vs Short OI + Imbalance for a Single Market

Change `MARKET` to explore any symbol.

In [ ]:
MARKET = "ETH/USD"
df_m = daily_oi[daily_oi["symbol"] == MARKET].sort_values("date")

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Long vs Short OI (USD)", "OI Imbalance (Long − Short)"],
    vertical_spacing=0.12,
)
fig.add_trace(
    go.Scatter(x=df_m["date"], y=df_m["oi_long"], name="Long OI",
               fill="tozeroy", line=dict(color="#00cc96")),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=df_m["date"], y=df_m["oi_short"], name="Short OI",
               fill="tozeroy", line=dict(color="#ef553b")),
    row=1, col=1,
)
fig.add_trace(
    go.Bar(x=df_m["date"], y=df_m["oi_imbalance"], name="Imbalance",
           marker_color="#636efa"),
    row=2, col=1,
)
fig.update_layout(
    title=f"{MARKET} — Long vs Short OI",
    template="plotly_dark", hovermode="x unified", height=600,
)
fig.show()

## Chart 3: OI Breakdown by Market — Latest Snapshot (Stacked Bar)

In [ ]:
latest_date = daily_oi["date"].max()
latest = daily_oi[daily_oi["date"] == latest_date].sort_values("oi_total", ascending=False)

fig = px.bar(
    latest, x="symbol", y=["oi_long", "oi_short"],
    title=f"GMX Markets — OI Breakdown ({latest_date.date()})",
    labels={"value": "OI (USD)", "variable": "Side", "symbol": "Market"},
    barmode="stack", template="plotly_dark",
    color_discrete_map={"oi_long": "#00cc96", "oi_short": "#ef553b"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Chart 4: Monthly Average OI Heatmap (All Markets)

In [ ]:
monthly = (
    daily_oi.assign(month=daily_oi["date"].dt.to_period("M"))
    .groupby(["month", "symbol"])["oi_total"]
    .mean()
    .reset_index()
)
monthly["month_str"] = monthly["month"].astype(str)
pivot = monthly.pivot(index="symbol", columns="month_str", values="oi_total").fillna(0)

fig = px.imshow(
    pivot / 1e6,
    title="GMX Monthly Average OI per Market (USD millions)",
    labels={"color": "OI $M", "x": "Month", "y": "Market"},
    color_continuous_scale="Blues",
    template="plotly_dark",
    aspect="auto",
)
fig.update_layout(height=max(400, len(pivot) * 25))
fig.show()